# Challenge 3: Can you change the correlation?
Qiskit Fall Fest 2026 – IBM Quantum, Vishnu Institute of Technology

In [ ]:
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit.visualization import plot_histogram

try:
    from qiskit_aer import AerSimulator
    backend = AerSimulator()
except ImportError:
    from qiskit.providers.basic_provider import BasicSimulator
    backend = BasicSimulator()

SHOTS = 100

# Gate colours (same style as the reference screenshot: [fill, text])
circuit_style = {
    "displaycolor": {
        "h":       ["#003399", "#FFFFFF"],
        "cx":      ["#003399", "#FFFFFF"],
        "x":       ["#CC3300", "#FFFFFF"],   # the added gate stands out in red
        "measure": ["#A9A9A9", "#000000"],
    }
}

## 1. Original circuit (Bell state)

In [ ]:
qc_original = QuantumCircuit(2, 2)
qc_original.h(0)          # superposition on qubit 0
qc_original.cx(0, 1)      # entangle qubit 0 with qubit 1
qc_original.measure([0, 1], [0, 1])

qc_original.draw(output="mpl", style=circuit_style)

## 2. Prediction (before running the modified circuit)
- **Original circuit:** only `00` and `11` should appear (about 50/50). The qubits always agree.
- **Modified circuit:** I add an **X gate on qubit 1** right after the CNOT. This flips qubit 1's result, so I expect `01` and `10` to dominate and `00` / `11` to (almost) disappear. The qubits will still be perfectly correlated, but now they will always **disagree**.

## 3. Modified circuit (one added gate: X on qubit 1)

In [ ]:
qc_modified = QuantumCircuit(2, 2)
qc_modified.h(0)
qc_modified.cx(0, 1)
qc_modified.x(1)          # <-- the ONE added gate
qc_modified.measure([0, 1], [0, 1])

qc_modified.draw(output="mpl", style=circuit_style)

## 4. Run both circuits (100 shots each)

In [ ]:
job_orig = backend.run(transpile(qc_original, backend), shots=SHOTS)
counts_original = job_orig.result().get_counts()

job_mod = backend.run(transpile(qc_modified, backend), shots=SHOTS)
counts_modified = job_mod.result().get_counts()

print("Original circuit results:", counts_original)
print("Modified circuit results:", counts_modified)

## 5. Histograms

In [ ]:
plot_histogram(counts_original, title="Original circuit (Bell state)",
               color="#003399", figsize=(7, 5))

In [ ]:
plot_histogram(counts_modified, title="Modified circuit (H, CX, X)",
               color="#CC3300", figsize=(7, 5))

## 6. Side-by-side comparison

In [ ]:
plot_histogram([counts_original, counts_modified],
               legend=["Original", "Modified"],
               color=["#003399", "#CC3300"],
               title="Original vs Modified", figsize=(8, 5))

## 7. Explanation
The original circuit applies H to qubit 0, putting it in superposition, then a CNOT copies that outcome onto qubit 1. This gives the Bell state (|00⟩ + |11⟩)/√2, so the results are only `00` and `11`, roughly 50% each.

In the modified circuit I added an **X gate on qubit 1** after the CNOT. X flips |0⟩ ↔ |1⟩, so the state becomes (|01⟩ + |10⟩)/√2. Now `01` and `10` appear (about 50% each) and `00` / `11` vanish. The qubits remain entangled and perfectly correlated, but the correlation changed from *same* to *opposite*. Small deviations from 50/50 are just shot noise from using only 100 shots.

**Gate count:** H, CX, X = 3 gates (≤ 4), 2 qubits, both measured, 100 shots.